[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/14_model_evaluation/90_exercises.ipynb)

# Bài tập — Chương 14: Model Evaluation (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/14_model_evaluation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics as skm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import make_customer_churn, make_fraud_transactions
from ml_course.utils import repo_path

rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** Viết công thức precision, recall, specificity, F1, $F_\beta$, MCC; định nghĩa TPR, FPR; ý nghĩa xác suất của ROC-AUC; công thức MAE, RMSE, $R^2$, WAPE, pinball loss.

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Giám đốc nói: "Model phát hiện gian lận có ROC-AUC 0.95 và accuracy 99%, vậy là rất tốt rồi". Giải thích (không dùng công thức) vì sao hai con số này có thể không đủ, và bạn sẽ hỏi lại những gì.

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** Confusion matrix: TP = 40, FN = 60, FP = 10, TN = 890. Tính accuracy, precision, recall, specificity, F1, MCC.

In [ ]:
TP, FN, FP, TN = 40, 60, 10, 890
### START CODE HERE ### (≈ 6 dòng)
acc = None
prec = None
rec = None
spec = None
f1 = None
mcc = None
### END CODE HERE ###

In [ ]:
y_t = np.r_[np.ones(TP + FN), np.zeros(FP + TN)]
y_p = np.r_[np.ones(TP), np.zeros(FN), np.ones(FP), np.zeros(TN)]
check_close(acc, 0.93, message="accuracy")
check_close((prec, rec, spec), (0.8, 0.4, 890 / 900), message="precision, recall, specificity")
check_close(f1, skm.f1_score(y_t, y_p), message="F1")
check_close(mcc, skm.matthews_corrcoef(y_t, y_p), message="MCC")

**Câu 4.** 3 positive có điểm `[0.9, 0.6, 0.35]`, 4 negative có điểm `[0.8, 0.4, 0.3, 0.1]`. Tính ROC-AUC bằng định nghĩa $P(s^+ > s^-)$ (đếm cặp).

In [ ]:
s_pos = np.array([0.9, 0.6, 0.35])
s_neg = np.array([0.8, 0.4, 0.3, 0.1])
### START CODE HERE ### (≈ 1 dòng)
auc_pairs = None
### END CODE HERE ###

In [ ]:
check_close(auc_pairs, 9 / 12, message="AUC = số cặp xếp đúng / tổng số cặp")
check_close(auc_pairs, skm.roc_auc_score(np.r_[np.ones(3), np.zeros(4)], np.r_[s_pos, s_neg]), message="khớp sklearn")

## L3 — Code

**Bài 5.** Cài đặt `roc_auc_rank(y, s)` bằng công thức **Mann–Whitney** với rank (xử lý hoà bằng rank trung bình — `scipy.stats.rankdata`): $\text{AUC} = \frac{R_+ - n_+(n_+ + 1)/2}{n_+ n_-}$, với $R_+$ là tổng rank của các positive. Độ phức tạp $O(n\log n)$.

In [ ]:
from scipy.stats import rankdata


def roc_auc_rank(y, s):
    ### START CODE HERE ### (≈ 4 dòng)
    ranks = None
    n_pos = None
    n_neg = None
    ### END CODE HERE ###

In [ ]:
y5 = rng.integers(0, 2, 5000)
s5 = np.round(rng.normal(y5 * 0.8, 1.0), 1)                     # làm tròn → nhiều điểm hoà
check_close(roc_auc_rank(y5, s5), skm.roc_auc_score(y5, s5), tol=1e-12, message="khớp sklearn (kể cả khi hoà)")

**Bài 6.** Viết `precision_recall_at_k(y, s, k)` và `best_threshold_for_cost(y, s, cost_fp, cost_fn)` (thử mọi ngưỡng là các giá trị điểm số duy nhất, dự đoán positive nếu $s \ge t$; trả về `(threshold, cost)` nhỏ nhất).

In [ ]:
def precision_recall_at_k(y, s, k):
    ### START CODE HERE ### (≈ 3 dòng)
    top = None
    tp = None
    ### END CODE HERE ###


def best_threshold_for_cost(y, s, cost_fp, cost_fn):
    ### START CODE HERE ### (≈ 5 dòng)
    thresholds = None
    costs = None
    i = None
    ### END CODE HERE ###

In [ ]:
y6 = np.array([1, 0, 1, 0, 0, 1, 0, 0])
s6 = np.array([0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.2, 0.1])
check_close(precision_recall_at_k(y6, s6, 3), (2 / 3, 2 / 3), message="P@3, R@3")
t6, c6 = best_threshold_for_cost(y6, s6, cost_fp=1, cost_fn=10)
check_close(t6, 0.4, message="FN đắt → hạ ngưỡng để bắt cả 3 positive")
check_close(c6, 3, message="chi phí = 3 FP")

## L4 — Debug

**Bài 7.** Đoạn đánh giá sau có **3 lỗi**: (1) tính ROC-AUC trên **nhãn dự đoán** 0/1 thay vì điểm số; (2) chọn ngưỡng tối ưu F1 trên **tập test** rồi báo cáo F1 cũng trên test (lạc quan); (3) dùng `average="micro"` cho F1 nhị phân mất cân bằng (= accuracy). Sửa: chia train/valid/test, chọn ngưỡng trên valid, báo cáo `test_auc` (từ điểm số) và `test_f1` (F1 lớp positive tại ngưỡng đã chọn).

In [ ]:
fraud = make_fraud_transactions(n=30000, fraud_rate=0.03, seed=5)
feats = ["amount_vnd", "hour", "distance_from_home_km", "is_foreign", "n_tx_last_24h", "card_age_days"]
Xf = fraud[feats].astype(float).assign(amount_vnd=lambda d: np.log(d.amount_vnd))
yf = fraud["is_fraud"].to_numpy()
### START CODE HERE ### (≈ 10 dòng)
X_tr, X_te, y_tr, y_te = train_test_split(Xf, yf, test_size=0.3, random_state=0, stratify=yf)
model = LogisticRegression(max_iter=2000).fit(X_tr, y_tr)
p_te = model.predict_proba(X_te)[:, 1]
test_auc = skm.roc_auc_score(y_te, (p_te >= 0.5).astype(int))                        # 🐛 (1)
ts = np.linspace(0.01, 0.9, 90)
best_t = ts[np.argmax([skm.f1_score(y_te, p_te >= t, average="micro") for t in ts])]   # 🐛 (2), (3)
test_f1 = skm.f1_score(y_te, p_te >= best_t, average="micro")                         # 🐛 (3)
### END CODE HERE ###
print(f"ngưỡng (chọn trên valid) = {best_t:.2f} | ROC-AUC test = {test_auc:.3f} | F1 test = {test_f1:.3f}")

In [ ]:
check_greater(test_auc, 0.75, "ROC-AUC tính từ điểm số (không phải nhãn 0/1)")
check_less(test_f1, 0.6, "F1 lớp positive (không phải micro-F1 ≈ accuracy)")
check_true(abs(test_auc - skm.roc_auc_score(y_te, (p_te >= 0.5).astype(int))) > 0.1, "AUC từ nhãn 0/1 khác hẳn AUC từ điểm số")

## L5 — Phân tích

**Bài 8.** Với điểm số của model trên `fraud` (Bài 7), mô phỏng tỉ lệ positive giảm từ 10% xuống 0.5% bằng cách **lấy mẫu lại** positive (giữ nguyên toàn bộ negative). Ghi ROC-AUC, PR-AUC, precision và recall tại ngưỡng cố định `best_t` vào DataFrame `prev` (cột `rate`, `roc_auc`, `pr_auc`, `precision`, `recall`) — mỗi tỉ lệ lặp lại 20 lần và lấy trung bình (vì khi positive rất hiếm, mỗi lần lấy mẫu cho metric rất khác nhau). Nhận xét metric nào thay đổi, metric nào không.

In [ ]:
pos_s, neg_s = p_te[y_te == 1], p_te[y_te == 0]
rows = []
### START CODE HERE ### (≈ 10 dòng)
# với mỗi rate: n_pos = số positive để đạt tỉ lệ rate (giữ toàn bộ neg_s); lặp 20 lần: lấy mẫu có hoàn lại n_pos điểm từ pos_s,
# tính 4 metric (precision/recall tại ngưỡng best_t); rows.append(dict trung bình 20 lần với các khoá rate, roc_auc, pr_auc, precision, recall)
pass
### END CODE HERE ###
prev = pd.DataFrame(rows)
prev.round(3)

In [ ]:
check_less(prev["roc_auc"].max() - prev["roc_auc"].min(), 0.03, "ROC-AUC gần như không đổi")
check_greater(prev["pr_auc"].iloc[0], 2 * prev["pr_auc"].iloc[-1], "PR-AUC giảm mạnh khi positive hiếm")
check_greater(prev["precision"].iloc[0], 2 * prev["precision"].iloc[-1], "precision tại ngưỡng cố định giảm mạnh")

✍️ Nhận xét:

## L6 — Thực tế

**Bài 9.** Đội chăm sóc khách hàng gọi được **150 khách/tuần** để giữ chân. Với `customer_churn.csv`: train Logistic Regression (one-hot + scale) và HistGradientBoosting; so sánh **Precision@150** và **Recall@150** trên test (25%, `random_state=0`, `stratify`), kèm **khoảng tin cậy 95%** (bootstrap 500 lần) cho hiệu P@150 (HistGB − LR). Lưu `p150` (dict tên → P@150) và `diff_ci` (tuple 2 đầu mút).

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(repo_path("data", "sample", "customer_churn.csv"))
Xc = pd.get_dummies(df.drop(columns=["customer_id", "churn"]), dtype=float)
yc = df["churn"].to_numpy()
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, random_state=0, stratify=yc)
### START CODE HERE ### (≈ 12 dòng)
scores = None     # dict "LR"/"HistGB" → xác suất churn trên X_test
p150 = None       # dict tên → precision@150 (dùng precision_recall_at_k của Bài 6)
diff_ci = None    # (2.5%, 97.5%) của 500 hiệu P@150 (HistGB − LR) trên cùng các mẫu bootstrap của tập test
### END CODE HERE ###
print({k: round(v, 3) for k, v in p150.items()}, "| R@150:", {k: round(precision_recall_at_k(yc_te, s, 150)[1], 3) for k, s in scores.items()})
print(f"95% CI cho P@150 (HistGB − LR): [{diff_ci[0]:+.3f}, {diff_ci[1]:+.3f}]")

In [ ]:
check_greater(min(p150.values()), 2 * yc_te.mean(), "P@150 cao hơn hẳn tỉ lệ churn (gọi ngẫu nhiên)")
check_true(diff_ci[0] < diff_ci[1], "khoảng tin cậy hợp lệ")

✍️ Kết luận cho đội chăm sóc khách hàng: chọn model nào? Có khác biệt đáng kể không?

## L7 — Nâng cao

**Bài 10.** **Khoảng tin cậy cho ROC-AUC bằng công thức Hanley & McNeil (1982):** $SE = \sqrt{\frac{A(1 - A) + (n_+ - 1)(Q_1 - A^2) + (n_- - 1)(Q_2 - A^2)}{n_+ n_-}}$ với $Q_1 = \frac{A}{2 - A}$, $Q_2 = \frac{2A^2}{1 + A}$. Cài đặt `hanley_mcneil_se(auc, n_pos, n_neg)` và so sánh khoảng $A \pm 1.96\,SE$ với khoảng bootstrap (1000 lần) cho ROC-AUC của LR trên churn (Bài 9).

In [ ]:
def hanley_mcneil_se(auc, n_pos, n_neg):
    ### START CODE HERE ### (≈ 3 dòng)
    q1 = None
    q2 = None
    ### END CODE HERE ###

In [ ]:
s_lr = scores["LR"]
A = skm.roc_auc_score(yc_te, s_lr)
se = hanley_mcneil_se(A, yc_te.sum(), (yc_te == 0).sum())
boot = []
r = np.random.default_rng(2)
for _ in range(1000):
    idx = r.integers(0, len(yc_te), len(yc_te))
    boot.append(skm.roc_auc_score(yc_te[idx], s_lr[idx]))
lo_b, hi_b = np.quantile(boot, [0.025, 0.975])
print(f"AUC = {A:.4f} | Hanley–McNeil: [{A - 1.96 * se:.4f}, {A + 1.96 * se:.4f}] | bootstrap: [{lo_b:.4f}, {hi_b:.4f}]")
check_close(2 * 1.96 * se, hi_b - lo_b, tol=0.25 * (hi_b - lo_b), message="độ rộng khoảng tin cậy hai cách gần nhau (±25%)")